# GoogleColabでのこのワークフローを実行するコード
Google Colab上で資料のワークフローを再現するための、ステップバイステップの実装コードを提示します。
Colab特有のI/Oボトルネック（Google Driveからの直接読み込みによる遅延）を回避するため、データセットを一度Colabのローカルストレージ（/content/）に展開する手順を含めています。

# 1. 環境構築とドライブのマウント
クラウドインスタンスの環境差異によるサイレントな依存関係の競合を防ぐため、セッション開始時に必要なライブラリのバージョンを固定してインストールします。

In [1]:
# Google Driveのマウント（チェックポイントの永続化とZIPデータの取得用）
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# pythonパッケージのインストール

!pip install libreyolo albumentations sahi numpy opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 132.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [libreyolo]


In [3]:
# I/Oボトルネック回避のため、Drive上のデータセット(ZIP)をColabのローカルにコピーして解凍
import os
import shutil

zip_path = '/content/drive/MyDrive/projects/libreyolo/dataset.zip'  # 環境に合わせて変更してください。
local_extract_path = '/content/dataset'

if not os.path.exists(local_extract_path):
    # os.makedirs(local_extract_path)

    shutil.unpack_archive(zip_path, "/content")
    print("データセットをローカルストレージに展開しました。")

データセットをローカルストレージに展開しました。


In [15]:
#
# モデル、テストフォルダ、テスト画像の設定
#
from libreyolo import LibreYOLO
import shutil
import os

# model = LibreYOLO("LibreYOLO9t.pt")
model = LibreYOLO("LibreRFDETRn.pt")

# 学習済みのベストモデル
# best_model = LibreYOLO('/content/drive/MyDrive/projects/libreyolo/libreyolo_runs/rf_LibreYOLO9t_model/weights/best.pt')
best_model = LibreYOLO('/content/drive/MyDrive/projects/libreyolo/libreRFDETRn_runs/rf_LibreRFDETRn_model/weights/best.pt')

src_root="/content/dataset/test/images/"
SAMPLE_IMAGE="sc_video1_part07-00028.jpg"

# 既存の出力先があればクリーンアップ
if os.path.exists("/content/output_images"):
    shutil.rmtree("/content/output_images")
os.makedirs("/content/output_images")
if os.path.exists("/content/drive/MyDrive/projects/libreyolo/outimg_dir"):
    shutil.rmtree("/content/drive/MyDrive/projects/libreyolo/outimg_dir")
os.makedirs("/content/drive/MyDrive/projects/libreyolo/outimg_dir")



## オリジナルモデルでの推論実行

In [5]:
# 学習前モデル（オリジナルモデル）での検出精度の確認
import cv2
import shutil
import glob
from libreyolo import LibreYOLO

# NMS（Non-Maximum Suppression）の簡易実装用のダミー関数
# ※ 実際には torchvision.ops.nms などを利用します
def non_max_suppression(boxes, iou_threshold):
    # 重複するバウンディングボックスを統合する処理
    return boxes


# テスト画像の読み込み
frame = cv2.imread(src_root+SAMPLE_IMAGE)
# テスト画像を学習済みモデルで推論
results = model.predict(frame) # LibreYOLOの予測

# 画像にBBOXを描画
color = (0, 255, 0)
thickness = 2  # 枠線の太さ
for box in results.boxes.xyxy:
    # 矩形（BBOX）の描画
    xmin, ymin, xmax, ymax = map(int, box.tolist())
    cv2.rectangle(frame, (xmin, ymin), (xmax, ymax), color, thickness)

# BBOX描画画像の出力
cv2.imwrite("/content/output_images/"+SAMPLE_IMAGE, frame)

shutil.copy("/content/output_images/"+SAMPLE_IMAGE, "/content/drive/MyDrive/projects/libreyolo/outimg_dir/"+"old_"+SAMPLE_IMAGE)

'/content/drive/MyDrive/projects/libreyolo/outimg_dir/old_sc_video1_part07-00015.jpg'

# 2. 学習データの水増し（Augmentation）の設定
合成データの代わりに、Albumentationsを用いて環境変化に対するモデルの耐性を高めるための変換処理を定義します。

In [ ]:
# import albumentations as A
#
# 資料に記載されている強力なデータ拡張パイプライン
# augmentation = A.Compose([
#     A.GaussianBlur(blur_limit=(3, 7), p=0.3),         # モーションブラー[cite: 1]
#     A.GaussNoise(var_limit=(10, 50), p=0.3),          # センサーノイズ[cite: 1]
#     A.ImageCompression(quality_lower=60, p=0.4),      # ストリーム圧縮劣化[cite: 1]
#     A.RandomBrightnessContrast(p=0.5),                # 照明の変化[cite: 1]
#     A.Perspective(scale=(0.02, 0.08), p=0.3),         # カメラアングルの変動[cite: 1]
# ])
#
# ※ 実際の学習時は、このaugmentationをデータローダーやLibreYOLOの設定ファイルに組み込みます。

/tmp/ipykernel_3160/397584232.py:6: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10, 50), p=0.3),          # センサーノイズ[cite: 1]
/tmp/ipykernel_3160/397584232.py:7: UserWarning: Argument(s) 'quality_lower' are not valid for transform ImageCompression
  A.ImageCompression(quality_lower=60, p=0.4),      # ストリーム圧縮劣化[cite: 1]


# 3. 学習データの水増し
LibreYOLOの統合APIを使用し、Vision TransformerベースのRF-DETRモデルを学習させます。混合精度学習（AMP）を有効にしてGPUメモリの消費を抑えます。

In [7]:
import glob
import os
import shutil
import albumentations as A
import cv2
import yaml

# 1. Albumentationsの拡張パイプライン定義
transform = A.Compose(
    [
        A.HorizontalFlip(p=0.5),
        A.RandomBrightnessContrast(p=0.5),
        A.ShiftScaleRotate(
            shift_limit=0.05,
            scale_limit=0.1,
            rotate_limit=15,
            p=0.5,
            border_mode=cv2.BORDER_CONSTANT,
        ),
        A.GaussianBlur(blur_limit=(3, 5), p=0.2),
    ],
    bbox_params=A.BboxParams(
        format="yolo", label_fields=["class_labels"], min_visibility=0.3
    ),
)


def prepare_augmented_yolo_dataset(
    xsrc_root, xout_root, class_names, num_aug=3
):
    # 既存の出力先があればクリーンアップ
    if os.path.exists(xout_root):
        shutil.rmtree(xout_root)

    train_img_src = os.path.join(xsrc_root, "train", "images")
    train_lbl_src = os.path.join(xsrc_root, "train", "labels")
    val_img_src = os.path.join(xsrc_root, "valid", "images")
    val_lbl_src = os.path.join(xsrc_root, "valid", "labels")
    test_img_src = os.path.join(xsrc_root, "test", "images")
    test_lbl_src = os.path.join(xsrc_root, "test", "labels")

    train_img_out = os.path.join(xout_root, "train", "images")
    train_lbl_out = os.path.join(xout_root, "train", "labels")
    val_img_out = os.path.join(xout_root, "valid", "images")
    val_lbl_out = os.path.join(xout_root, "valid", "labels")
    test_img_out = os.path.join(xout_root, "test", "images")
    test_lbl_out = os.path.join(xout_root, "test", "labels")

    os.makedirs(train_img_out, exist_ok=True)
    os.makedirs(train_lbl_out, exist_ok=True)

    # --- 1. 学習データ（train）のコピー＆水増し ---
    img_files = glob.glob(os.path.join(train_img_src, "*.jpg"))
    print(f"学習データ検出数: {len(img_files)}枚")

    for img_path in img_files:
        base_name = os.path.splitext(os.path.basename(img_path))[0]
        label_path = os.path.join(train_lbl_src, f"{base_name}.txt")

        # 元データのコピー
        shutil.copy2(img_path, os.path.join(train_img_out, f"{base_name}.jpg"))
        if os.path.exists(label_path):
            shutil.copy2(
                label_path, os.path.join(train_lbl_out, f"{base_name}.txt")
            )

        # 画像とBBoxの読み込み
        image = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
        bboxes, class_labels = [], []
        if os.path.exists(label_path):
            with open(label_path, "r") as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        # int(float(...)) にして元データに小数が入っていても整数化
                        class_labels.append(int(float(parts[0])))
                        bboxes.append([float(x) for x in parts[1:5]])

        # 指定回数だけ水増し生成
        for i in range(num_aug):
            transformed = transform(
                image=image, bboxes=bboxes, class_labels=class_labels
            )
            aug_img = cv2.cvtColor(transformed["image"], cv2.COLOR_RGB2BGR)

            cv2.imwrite(
                os.path.join(train_img_out, f"{base_name}_aug{i}.jpg"), aug_img
            )

            with open(
                os.path.join(train_lbl_out, f"{base_name}_aug{i}.txt"), "w"
            ) as f:
                for cls_id, bbox in zip(
                    transformed["class_labels"], transformed["bboxes"]
                ):
                    bbox_str = " ".join(f"{coord:.6f}" for coord in bbox)
                    # 【修正】必ず int(cls_id) で整数として書き出す
                    f.write(f"{int(cls_id)} {bbox_str}\n")

    # --- 2. 検証データ（valid）・テストデータ（test）のコピー ---
    if os.path.exists(val_img_src):
        shutil.copytree(val_img_src, val_img_out, dirs_exist_ok=True)
    if os.path.exists(val_lbl_src):
        shutil.copytree(val_lbl_src, val_lbl_out, dirs_exist_ok=True)
    if os.path.exists(test_img_src):
        shutil.copytree(test_img_src, test_img_out, dirs_exist_ok=True)
    if os.path.exists(test_lbl_src):
        shutil.copytree(test_lbl_src, test_lbl_out, dirs_exist_ok=True)

    # --- 3. dataset.yaml の生成 ---
    yaml_data = {
        "path": os.path.abspath(xout_root),
        "train": "train/images",
        "val": "valid/images",
        "test": "test/images",
        "nc": len(class_names),
        "names": {i: name for i, name in enumerate(class_names)},
    }

    # 学習コードの指定に合わせて dataset.yaml で出力
    yaml_path = os.path.join(xout_root, "dataset.yaml")
    with open(yaml_path, "w", encoding="utf-8") as f:
        yaml.dump(
            yaml_data,
            f,
            default_flow_style=False,
            allow_unicode=True,
            sort_keys=False,
        )

    print(f"完了！設定ファイルを出力しました: {yaml_path}")


# === 実行 ===
if __name__ == "__main__":
    orig_yaml_path = "/content/dataset/dataset.yaml"
    if os.path.exists(orig_yaml_path):
        with open(orig_yaml_path, "r", encoding="utf-8") as f:
            orig_cfg = yaml.safe_load(f)
        if isinstance(orig_cfg.get("names"), dict):
            CLASSES = [
                orig_cfg["names"][k] for k in sorted(orig_cfg["names"].keys())
            ]
        else:
            CLASSES = orig_cfg.get("names", ["class0"])
    else:
        CLASSES = ["class0"]  # 必要に応じて書き換えてください

    prepare_augmented_yolo_dataset(
        xsrc_root="/content/dataset",
        xout_root="/content/dataset_aug",
        class_names=CLASSES,
        num_aug=3,
    )


/usr/local/lib/python3.13/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


学習データ検出数: 1262枚
完了！設定ファイルを出力しました: /content/dataset_aug/dataset.yaml


## RF-DETRモデルの学習

In [ ]:
#*******************
#   modelの学習
#*******************

local_extract_path1 = '/content/dataset_aug'

# 学習の実行
# プロジェクトの保存先をGoogle Driveに指定し、Spot VM（Colabセッション）中断時の消失を防ぐ
results = model.train(
    data=f'{local_extract_path1}/dataset.yaml', # ローカルに展開したデータセットを指定
    epochs=100,
    batch=16,
    amp=True,  # 混合精度学習(FP16)を有効化[cite: 1]
    patience=25,  # 20〜30エポック改善がない場合の早期終了(Early Stopping)[cite: 1]
    project='/content/drive/MyDrive/projects/libreyolo/libreRFDETRn_runs', # チェックポイントの保存先をDriveへ
    name='rf_LibreRFDETRn_model'
)

INFO:libreyolo.models.rfdetr.matcher:RF-DETR assignment backend: scipy
INFO:libreyolo.training.trainer:Epoch 1 - Average loss: 4.2696
INFO:libreyolo.training.trainer:Running validation for epoch 1
INFO:libreyolo.data.dataset:Loading 158 YOLO annotations from valid/labels...
INFO:libreyolo.data.dataset:Loaded 158 YOLO annotations from valid/labels in 0.03s
INFO:libreyolo.data.yolo_coco_api:Building COCO API for 158 images...
INFO:libreyolo.data.yolo_coco_api:  Loaded 1573 annotations across 158 images
INFO:libreyolo.validation.coco_evaluator:COCO eval backend: pycocotools ?


Running per image evaluation...
Evaluate annotation type *bbox*


INFO:libreyolo.training.trainer:Validation - mAP50: 0.7801, mAP50-95: 0.5604


DONE (t=0.75s).
Accumulating evaluation results...
DONE (t=0.04s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.560
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.780
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.622
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.032
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.444
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.748
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.078
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.505
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.641
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.049
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.555
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= lar

INFO:libreyolo.training.trainer:New best model saved - Epoch 1: metrics/mAP50-95=0.5604
INFO:libreyolo.training.trainer:Checkpoint saved: /content/drive/MyDrive/projects/libreyolo/libreRFDETRn_runs/rf_LibreRFDETRn_model/weights/last.pt
INFO:libreyolo.training.trainer:Epoch 2 - Average loss: 4.0441
INFO:libreyolo.training.trainer:Running validation for epoch 2
INFO:libreyolo.validation.coco_evaluator:COCO eval backend: pycocotools ?


Running per image evaluation...
Evaluate annotation type *bbox*


... (239 outputs left out)

Running per image evaluation...
Evaluate annotation type *bbox*


INFO:libreyolo.training.trainer:Validation - mAP50: 0.7813, mAP50-95: 0.6248


DONE (t=0.73s).
Accumulating evaluation results...
DONE (t=0.04s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.625
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.781
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.680
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.040
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.581
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.789
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.083
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.544
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.698
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.069
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.647
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= lar

INFO:libreyolo.training.trainer:Checkpoint saved: /content/drive/MyDrive/projects/libreyolo/libreRFDETRn_runs/rf_LibreRFDETRn_model/weights/last.pt
INFO:libreyolo.training.trainer:Early stopping triggered after 62 epochs (patience=25, no improvement for 25 epochs)
INFO:libreyolo.training.trainer:Training complete in 1.36 hours


# 4. 信頼度しきい値の自動キャリブレーション
固定のしきい値による誤報を防ぐため、F1スコアが最大となる最適なしきい値を探索します。

In [8]:
import numpy as np

CONF_THRESHOLDS = np.arange(0.01, 0.96, 0.01)
f1_scores = []


# 各しきい値でのPrecision, Recall, F1スコアの計算
for conf in CONF_THRESHOLDS:
    val_results = best_model.val(data=f'{local_extract_path}/dataset.yaml', conf=conf)

    # BoxImageMetrics 内の配列から平均値を取得
    p = np.mean(val_results.box.p)
    r = np.mean(val_results.box.r)

    # F1スコアの計算（PrecisionとRecallの調和平均）
    f1 = 2 * p * r / (p + r + 1e-9)
    f1_scores.append(f1)

# F1スコアが最も高くなるしきい値を選択
best_f1_conf = CONF_THRESHOLDS[np.argmax(f1_scores)]

print(f"Deployment向けの最適なしきい値: {best_f1_conf}")

Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.73s).
Accumulating evaluation results...
DONE (t=0.04s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.628
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.797
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.685
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.045
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.575
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.798
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.083
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.543
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.696
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.074
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=medium | maxDets=100

In [9]:
#
# 単一画像による学習後モデルの精度確認のための推論
#
import cv2
import shutil

# NMS（Non-Maximum Suppression）の簡易実装用のダミー関数
# ※ 実際には torchvision.ops.nms などを利用します
def non_max_suppression(boxes, iou_threshold):
    # 重複するバウンディングボックスを統合する処理
    return boxes


# テスト画像の読み込み
frame = cv2.imread(src_root+SAMPLE_IMAGE)
# テスト画像を学習済みモデルで推論
results = best_model.predict(frame) # LibreYOLOの予測
# results = best_model.predict(frame, conf=best_f1_conf)[0]
print(results.boxes.xyxy)

# 画像にBBOXを描画
color = (0, 255, 0)
thickness = 2  # 枠線の太さ
for box in results.boxes.xyxy:
    # 矩形（BBOX）の描画
    xmin, ymin, xmax, ymax = map(int, box.tolist())
    cv2.rectangle(frame, (xmin, ymin), (xmax, ymax), color, thickness)

# BBOX描画画像の出力
cv2.imwrite("/content/output_images/"+SAMPLE_IMAGE, frame)
shutil.copy("/content/output_images/"+SAMPLE_IMAGE, "/content/drive/MyDrive/projects/libreyolo/outimg_dir/"+"new_"+SAMPLE_IMAGE)


tensor([[ 942.8895,  556.2674, 1017.2275,  767.3210],
        [1602.1935,  718.0623, 1714.4529,  933.9389],
        [ 411.0396,  175.9585,  506.6741,  303.1824],
        [ 532.4507,  712.8348,  622.7242,  914.3953],
        [1499.9865,  304.7923, 1560.2465,  465.8406],
        [1853.7340,  226.2486, 1918.7662,  380.5926]])


'/content/drive/MyDrive/projects/libreyolo/outimg_dir/new_sc_video1_part07-00015.jpg'

In [10]:
# testディレクトリの画像を一括検出処理
#
import os
import glob
import shutil
import cv2
from tqdm import tqdm

def predict(model, image, conf=0.25):
    """OpenCV(BGR)画像をRGB変換して推論し、[[x1, y1, x2, y2, score, cls_id], ...] を返す"""
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    res = model.predict(image_rgb, conf=conf)[0]
    if res.boxes is None or len(res.boxes) == 0:
        return []
    return [
        box.tolist() + [float(score), int(cls_id)]
        for box, score, cls_id in zip(res.boxes.xyxy, res.boxes.conf, res.boxes.cls)
    ]

# ==========================================
# パス・パラメータ設定
# ==========================================
TEST_IMG_DIR    = "/content/dataset/test/images"
LOCAL_OUT_DIR   = "/content/output_images"
DRIVE_OUT_DIR   = "/content/drive/MyDrive/projects/libreyolo/new_outimg_dir"
CONF_THRESHOLD  = 0.11  # 必要に応じて調整、0.25

# 出力ディレクトリの初期化
if os.path.exists(LOCAL_OUT_DIR):
    shutil.rmtree(LOCAL_OUT_DIR)
os.makedirs(LOCAL_OUT_DIR)

if os.path.exists(DRIVE_OUT_DIR):
    shutil.rmtree(DRIVE_OUT_DIR)

# 対象画像の取得（.jpg, .jpeg, .png に対応）
extensions = ("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG")
image_paths = []
for ext in extensions:
    image_paths.extend(glob.glob(os.path.join(TEST_IMG_DIR, ext)))
image_paths = sorted(image_paths)

print(f"対象画像数: {len(image_paths)} 枚")

# ==========================================
# 一括推論とBBOX描画
# ==========================================
box_color  = (0, 255, 0)       # 緑
text_color = (255, 255, 255)   # 白
thickness  = 2

total_detections = 0

for img_path in tqdm(image_paths, desc="推論実行中"):
    frame = cv2.imread(img_path)
    if frame is None:
        print(f"読み込みスキップ: {img_path}")
        continue

    # 推論（内部でRGB変換）
    results = predict(best_model, frame, conf=CONF_THRESHOLD)
    total_detections += len(results)

    # BBOXとスコアの描画（元のBGR画像に描画）
    for det in results:
        xmin, ymin, xmax, ymax = map(int, det[:4])
        score, cls_id = det[4], det[5]
        cv2.rectangle(frame, (xmin, ymin), (xmax, ymax), box_color, thickness)
        cv2.putText(
            frame,
            f"cls:{cls_id} {score:.2f}",
            (xmin, max(ymin - 5, 10)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            text_color,
            1
        )

    # ローカルへ保存
    out_path = os.path.join(LOCAL_OUT_DIR, os.path.basename(img_path))
    cv2.imwrite(out_path, frame)

# 最後にまとめて Google Drive へコピー
shutil.copytree(LOCAL_OUT_DIR, DRIVE_OUT_DIR)

print(f"完了: {len(image_paths)} 枚を処理（総検出数: {total_detections}）")
print(f"保存先: {DRIVE_OUT_DIR}")

対象画像数: 158 枚


推論実行中:   0%|          | 0/158 [00:00<?, ?it/s]/tmp/ipykernel_715/2245129701.py:12: DeprecationWarning: results[0], results[-1] and `for r in results` on a single-image Results return the whole result with all its detections, as for the list predict() returns for several images. Earlier releases selected detections. Select detections with a slice, mask or index array (results[:1], results[results.boxes.conf > 0.5]) or use results.boxes[i].
  res = model.predict(image_rgb, conf=conf)[0]
推論実行中: 100%|██████████| 158/158 [00:05<00:00, 30.06it/s]


完了: 158 枚を処理（総検出数: 2288）
保存先: /content/drive/MyDrive/projects/libreyolo/new_outimg_dir


In [ ]:
# 画像チェック用コード
import cv2

NG_IMAGE = "/content/dataset/test/images/sc_video1_part08-00014.jpg"
frame_bgr = cv2.imread(NG_IMAGE)
frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

# パターン1: ファイルパスを直接渡す（ライブラリ側の標準読み込み）
res_path = best_model.predict(NG_IMAGE, conf=0.05)
print("1. パス渡し (conf=0.05):", len(res_path.boxes), res_path.boxes.conf[:5] if len(res_path.boxes) else [])

# パターン2: RGB変換して渡す
res_rgb = best_model.predict(frame_rgb, conf=0.05)
print("2. RGB渡し  (conf=0.05):", len(res_rgb.boxes), res_rgb.boxes.conf[:5] if len(res_rgb.boxes) else [])

# パターン3: 今のままBGRで、confだけ下げて渡す
res_bgr = best_model.predict(frame_bgr, conf=0.05)
print("3. BGR渡し  (conf=0.05):", len(res_bgr.boxes), res_bgr.boxes.conf[:5] if len(res_bgr.boxes) else [])

1. パス渡し (conf=0.05): 10 tensor([0.1869, 0.1593, 0.1542, 0.1476, 0.1289])
2. RGB渡し  (conf=0.05): 32 tensor([0.8969, 0.7595, 0.7273, 0.7081, 0.6678])
3. BGR渡し  (conf=0.05): 10 tensor([0.1869, 0.1593, 0.1542, 0.1476, 0.1289])


# 5. SAHIを用いた小物体検出（推論）
広角映像内の小さな物体（ナイフや携帯電話など）を検出するため、入力フレームをタイル状に分割・推論・結合するSAHIを実装します。

In [16]:
# SAHIの改善点（右端・下端のカバー、タイル端の欠け抑制、IOU、IOSベースのNMM統合、画像サイズに応じた自動切替）と信頼度の下限設定（`safe_conf`）も含め、
# ペナルティ後の再フィルタリング、細長ボックスの除外、および描画テキストの色を白（`(255, 255, 255)`）に変更した全体のコードです。
#
# 使い分けの目安
# RF-DETRで中〜近距離の映像: USE_SAHI = False
# 遠距離・俯瞰で対象物が小さい映像: USE_SAHI = True、POSTPROCESS = "nms_iou"
# 対象物同士が密集しておらず、境界分断を繋ぎたい場合: USE_SAHI = True、POSTPROCESS = "nmm_ios"、MATCH_THRESHOLD = 0.5

import os
import shutil
import cv2
import numpy as np


def get_slice_coords(length, tile_size, step):
    """画像の右端・下端まで漏れなくカバーするスライス開始座標を生成"""
    if length <= tile_size:
        return [0]
    coords = list(range(0, length - tile_size + 1, step))
    if coords[-1] + tile_size < length:
        coords.append(length - tile_size)
    return coords


def compute_iou(box1, box2):
    """2つのボックスの IoU (Intersection over Union) を計算"""
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    inter_w = max(0.0, x2 - x1)
    inter_h = max(0.0, y2 - y1)
    inter_area = inter_w * inter_h
    if inter_area == 0:
        return 0.0

    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union_area = area1 + area2 - inter_area

    return inter_area / (union_area + 1e-6)


def compute_ios(box1, box2):
    """2つのボックスの IOS (Intersection over Smaller) を計算"""
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    inter_w = max(0.0, x2 - x1)
    inter_h = max(0.0, y2 - y1)
    inter_area = inter_w * inter_h
    if inter_area == 0:
        return 0.0

    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    smaller_area = min(area1, area2)

    return inter_area / (smaller_area + 1e-6)


def greedy_nms_iou(detections, iou_threshold=0.45):
    """クラスごとに IoU ベースの NMS を適用（推奨・デフォルト）"""
    if not detections:
        return []

    final_detections = []
    classes = set(d[5] for d in detections)

    for cls_id in classes:
        cls_dets = [d for d in detections if d[5] == cls_id]
        cls_dets.sort(key=lambda x: x[4], reverse=True)

        keep = []
        while cls_dets:
            best = cls_dets.pop(0)
            keep.append(best)
            cls_dets = [
                d for d in cls_dets
                if compute_iou(best[:4], d[:4]) < iou_threshold
            ]
        final_detections.extend(keep)

    return final_detections


def greedy_nmm_ios(detections, ios_threshold=0.5):
    """クラスごとに IOS ベースの NMM を適用（分断ボックスを外接矩形で統合）"""
    if not detections:
        return []

    final_detections = []
    classes = set(d[5] for d in detections)

    for cls_id in classes:
        cls_dets = [d for d in detections if d[5] == cls_id]
        cls_dets.sort(key=lambda x: x[4], reverse=True)

        merged = []
        used = [False] * len(cls_dets)

        for i in range(len(cls_dets)):
            if used[i]:
                continue
            base = list(cls_dets[i])
            used[i] = True

            for j in range(i + 1, len(cls_dets)):
                if used[j]:
                    continue
                if compute_ios(base[:4], cls_dets[j][:4]) >= ios_threshold:
                    base[0] = min(base[0], cls_dets[j][0])
                    base[1] = min(base[1], cls_dets[j][1])
                    base[2] = max(base[2], cls_dets[j][2])
                    base[3] = max(base[3], cls_dets[j][3])
                    used[j] = True

            merged.append(base)
        final_detections.extend(merged)

    return final_detections


def sahi_predict_improved(
    smodel,
    image,
    tile_size=640,
    overlap=0.3,
    conf=0.25,
    postprocess="nms_iou",  # "nms_iou"（推奨） または "nmm_ios"
    match_threshold=0.45,   # nms_iou の場合は IoU閾値、nmm_ios の場合は IOS閾値
    edge_margin=5,
    edge_penalty=0.6,
    max_box_ratio=0.4,      # 元画像の幅・高さの40%を超える巨大誤検出を除外
    include_full_image=True
):
    """RF-DETR / YOLO 両対応のSAHI推論"""
    H, W = image.shape[:2]
    step = int(tile_size * (1 - overlap))
    all_detections = []

    def is_valid_box(x1, y1, x2, y2):
        bw, bh = x2 - x1, y2 - y1
        # 極端に細長いエッジ誤検出を除外
        if bw < 10 or (bh / (bw + 1e-6)) > 6.0:
            return False
        # 画面に対して不自然に巨大な誤検出を除外
        if bw > W * max_box_ratio or bh > H * max_box_ratio:
            return False
        return True

    # 1. フル画像での推論（大きい物体の全身ボックスを取得）
    if include_full_image:
        full_results = smodel.predict(image, conf=conf)[0]
        if full_results.boxes is not None and len(full_results.boxes) > 0:
            for box, score, cls_id in zip(
                full_results.boxes.xyxy,
                full_results.boxes.conf,
                full_results.boxes.cls
            ):
                x1, y1, x2, y2 = box.tolist()
                if is_valid_box(x1, y1, x2, y2):
                    all_detections.append([x1, y1, x2, y2, float(score), int(cls_id)])

    # 2. スライス（タイル）推論
    y_coords = get_slice_coords(H, tile_size, step)
    x_coords = get_slice_coords(W, tile_size, step)

    for y in y_coords:
        for x in x_coords:
            y_end = min(y + tile_size, H)
            x_end = min(x + tile_size, W)
            tile = image[y:y_end, x:x_end]
            tile_h, tile_w = tile.shape[:2]

            sahi_results = model.predict(tile, conf=conf)[0]
            if sahi_results.boxes is None or len(sahi_results.boxes) == 0:
                continue

            for box, score, cls_id in zip(
                sahi_results.boxes.xyxy,
                sahi_results.boxes.conf,
                sahi_results.boxes.cls
            ):
                x1, y1, x2, y2 = box.tolist()
                adj_score = float(score)

                # タイルの切れ目（元画像の端ではない境界）に接しているか判定
                touch_left   = (x > 0) and (x1 <= edge_margin)
                touch_top    = (y > 0) and (y1 <= edge_margin)
                touch_right  = (x_end < W) and (x2 >= tile_w - edge_margin)
                touch_bottom = (y_end < H) and (y2 >= tile_h - edge_margin)

                # 途中で切れている可能性が高いボックスは優先度を下げ、閾値未満なら除外
                if touch_left or touch_top or touch_right or touch_bottom:
                    adj_score *= edge_penalty
                    if adj_score < conf:
                        continue

                gx1, gy1, gx2, gy2 = x1 + x, y1 + y, x2 + x, y2 + y
                if is_valid_box(gx1, gy1, gx2, gy2):
                    all_detections.append([gx1, gy1, gx2, gy2, adj_score, int(cls_id)])

    # 3. 後処理の切り替え
    if postprocess == "nmm_ios":
        return greedy_nmm_ios(all_detections, ios_threshold=match_threshold)
    elif postprocess == "nms_iou":
        return greedy_nms_iou(all_detections, iou_threshold=match_threshold)
    else:
        raise ValueError(f"未対応のpostprocessです: {postprocess}")


def predict(
    pmodel,
    image,
    conf=0.25,
    use_sahi=False,
    tile_size=640,
    overlap=0.3,
    postprocess="nms_iou",
    match_threshold=0.45,
    max_box_ratio=0.4,
    include_full_image=True
):
    """
    通常推論とSAHI推論をパラメータで明示的に切り替える統合推論関数
    - use_sahi=False: 通常推論（中距離・近距離・RF-DETRの標準画角向け）
    - use_sahi=True : SAHI推論（広角・遠距離・俯瞰で対象物が小さい画角向け）
    """
    if use_sahi:
        return sahi_predict_improved(
            model=pmodel,
            image=image,
            tile_size=tile_size,
            overlap=overlap,
            conf=conf,
            postprocess=postprocess,
            match_threshold=match_threshold,
            max_box_ratio=max_box_ratio,
            include_full_image=include_full_image
        )

    # 通常推論
    res = pmodel.predict(image, conf=conf)[0]
    if res.boxes is None or len(res.boxes) == 0:
        return []
    return [
        box.tolist() + [float(score), int(cls_id)]
        for box, score, cls_id in zip(res.boxes.xyxy, res.boxes.conf, res.boxes.cls)
    ]


# ==========================================
# 実行パラメータ設定（ここを変更して制御）
# ==========================================
USE_SAHI        = False      # True: SAHI推論 / False: 通常推論
POSTPROCESS     = "nms_iou"  # "nms_iou"（推奨） または "nmm_ios"
MATCH_THRESHOLD = 0.45       # nms_iouなら0.45前後、nmm_iosなら0.5前後
TILE_SIZE       = 640
OVERLAP         = 0.3
MIN_CONF        = 0.20       # 信頼度の下限値（誤検出防止）


# テスト画像の読み込み
print(src_root+SAMPLE_IMAGE)
frame = cv2.imread(src_root+SAMPLE_IMAGE)

safe_conf = max(best_f1_conf, MIN_CONF)
print(f"use_sahi={USE_SAHI}, postprocess={POSTPROCESS}, conf={safe_conf:.2f}")

# 推論の実行
final_detections = predict(
    pmodel=best_model,
    image=frame,
    conf=safe_conf,
    use_sahi=USE_SAHI,
    tile_size=TILE_SIZE,
    overlap=OVERLAP,
    postprocess=POSTPROCESS,
    match_threshold=MATCH_THRESHOLD
)

# 画像にBBOX・クラス・信頼度を描画
box_color = (0, 255, 0)         # BBOXの色（緑）
text_color = (255, 255, 255)    # テキストの色（白）
thickness = 2

for det in final_detections:
    xmin, ymin, xmax, ymax = map(int, det[:4])
    score, cls_id = det[4], det[5]
    cv2.rectangle(frame, (xmin, ymin), (xmax, ymax), box_color, thickness)
    cv2.putText(
        frame,
        f"cls:{cls_id} {score:.2f}",
        (xmin, max(ymin - 5, 10)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.5,
        text_color,
        1
    )

cv2.imwrite("/content/drive/MyDrive/projects/libreyolo/outimg_dir/"+"sahi_"+SAMPLE_IMAGE, frame)


/content/dataset/test/images/sc_video1_part07-00028.jpg
use_sahi=False, postprocess=nms_iou, conf=0.20


True